# Figure 5: dCA1 spatial stability

Panels **5b–e and 5g–l**. Historical source filenames use Fig.4 numbering.

This notebook follows **Xu et al., manuscript dated 30 October 2026**. Data loading, model fitting, panel selection and plotting are shown below as executable steps. Shared scientific methods remain in `src/`; this notebook does not delegate the figure to `run_figure()`.

Start with the separate manuscript source-data files listed in `data/required_files.csv`. These are analysis source tables and frozen map/embedding arrays, rather than raw movies. The aligned-data computation examples are in `00_recompute_metrics.ipynb`.

Run cells from top to bottom. `REFIT_STATISTICS=True` refits the manuscript models; set it to `False` to use the frozen estimates. Outputs are saved separately from inputs.

## 1. Paths and execution parameters

Set `PPRH_DATA_DIR` or edit `DATA_ROOT`. The default refits statistical models. No raw neural resampling is triggered here.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "main.py").is_file() and (p / "src/paper.py").is_file())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import SVG, Image, display
from paper_io import read_table, save_panel as _save_panel, write_tables, p_symbol, write_run_record

DATA_ROOT = Path(os.environ.get("PPRH_DATA_DIR", str(ROOT / "data/input")))
OUTPUT_ROOT = Path(os.environ.get("PPRH_OUTPUT_DIR", str(ROOT / "outputs/notebooks")))
REFIT_STATISTICS = os.environ.get("PPRH_REFIT_STATISTICS", "1") == "1"
FORMATS = ("svg", "png")
DPI = 300

data_root = DATA_ROOT
output = OUTPUT_ROOT / "Fig5"
refit, formats, dpi = REFIT_STATISTICS, FORMATS, DPI
output.mkdir(parents=True, exist_ok=True)
if not data_root.is_dir():
    raise FileNotFoundError("Supply the separate manuscript inputs through DATA_ROOT; see data/README.md.")


## 2. Analysis contract

Within-day maps use condition-ordinal odd/even correct trials; map displays select D1, D3 and D5 while longitudinal models retain D1–D5. Single-neuron correlations are Pearson; PV correlations are Spearman with fixed 50-neuron sampling. Raw `r` is plotted and Fisher `z` is used for inference. Adjacent and D1-anchor day pairs are analyzed separately. Original clustering and movement-covariate choices are retained; no new across-day averaging is performed.

## 3. Notebook functions

The export function saves and previews each panel. Where needed, the source-table loader or annotation helper is defined here in full. Numerical estimators and shared plotting functions are imported in the relevant analysis cells.

In [ ]:
def save_panel(fig, output_root, panel, description, **kwargs):
    """Save one panel with the shared export method, then preview its result."""
    paths = _save_panel(fig, output_root, panel, description, **kwargs)
    svg = next((p for p in paths if p.endswith(".svg")), None)
    png = next((p for p in paths if p.endswith(".png")), None)
    if svg:
        display(SVG(filename=svg))
    elif png:
        display(Image(filename=png))
    return paths


In [ ]:
def _stability_ci_sources(data_root):
    names = {'within_pv_bin': 'Fig4_within_pv_bin.csv', 'within_pv': 'Fig4_within_pv.csv', 'within_single': 'Fig4_within_single.csv', 'inter_pv_bin': 'Fig4_inter_pv_bin.csv', 'inter_pv': 'Fig4_inter_pv.csv', 'inter_single': 'Fig4_inter_single.csv'}
    return {k: read_table(data_root, 'fig4/source_data/' + v) for k, v in names.items()}


## 4. Source data, statistics and panel-by-panel plots

Each panel cell names its source, contrast and display parameters. Use the current panel names below; historical filenames remain unchanged for traceability. Model fitting can take longer than figure rendering.

### Read the source data

In [ ]:
from figure_helpers import figure4 as h

sources = _stability_ci_sources(data_root)

statistics = read_table(data_root, 'fig4/stats/Fig4_LMM_all_contrasts.csv')


In [ ]:
display(statistics.head())


### Fit the manuscript models or retain the frozen estimates

In [ ]:
if refit:
    statistics, coefficients, _ = h.fit_all_models(sources, maxiter=1000)
    write_tables({'stability_ci_refitted_estimands': statistics, 'stability_ci_refitted_coefficients': coefficients}, Path(output) / 'statistics')


### Compute descriptive summaries with day retained

In [ ]:
descriptives = h.build_descriptive_tables(sources)

exports = []

within, adjacent = ({}, {})


### Prepare the next analysis step

In [ ]:
group = 'C'


### Read the source data

In [ ]:
day = 1

path = Path(data_root) / f'fig4/source_data/heatmaps/Fig4D_{group}_D{day}_CL_matrices.npz'

with np.load(path) as z:
    within[group, day] = {'Odd': z['map_a_display'], 'Even': z['map_b_display']}


### Read the source data

In [ ]:
day = 3

path = Path(data_root) / f'fig4/source_data/heatmaps/Fig4D_{group}_D{day}_CL_matrices.npz'

with np.load(path) as z:
    within[group, day] = {'Odd': z['map_a_display'], 'Even': z['map_b_display']}


### Read the source data

In [ ]:
day = 5

path = Path(data_root) / f'fig4/source_data/heatmaps/Fig4D_{group}_D{day}_CL_matrices.npz'

with np.load(path) as z:
    within[group, day] = {'Odd': z['map_a_display'], 'Even': z['map_b_display']}


### Read the source data

In [ ]:
a, b = (1, 2)

path = Path(data_root) / f'fig4/source_data/heatmaps/Fig4H_{group}_D{a}-D{b}_CL_matrices.npz'

with np.load(path) as z:
    adjacent[group, f'D{a}–D{b}'] = {'earlier': z['map_a_display'], 'later': z['map_b_display']}


### Read the source data

In [ ]:
a, b = (2, 3)

path = Path(data_root) / f'fig4/source_data/heatmaps/Fig4H_{group}_D{a}-D{b}_CL_matrices.npz'

with np.load(path) as z:
    adjacent[group, f'D{a}–D{b}'] = {'earlier': z['map_a_display'], 'later': z['map_b_display']}


### Read the source data

In [ ]:
a, b = (3, 4)

path = Path(data_root) / f'fig4/source_data/heatmaps/Fig4H_{group}_D{a}-D{b}_CL_matrices.npz'

with np.load(path) as z:
    adjacent[group, f'D{a}–D{b}'] = {'earlier': z['map_a_display'], 'later': z['map_b_display']}


### Read the source data

In [ ]:
a, b = (4, 5)

path = Path(data_root) / f'fig4/source_data/heatmaps/Fig4H_{group}_D{a}-D{b}_CL_matrices.npz'

with np.load(path) as z:
    adjacent[group, f'D{a}–D{b}'] = {'earlier': z['map_a_display'], 'later': z['map_b_display']}


### Prepare the next analysis step

In [ ]:
group = 'I'


### Read the source data

In [ ]:
day = 1

path = Path(data_root) / f'fig4/source_data/heatmaps/Fig4D_{group}_D{day}_CL_matrices.npz'

with np.load(path) as z:
    within[group, day] = {'Odd': z['map_a_display'], 'Even': z['map_b_display']}


### Read the source data

In [ ]:
day = 3

path = Path(data_root) / f'fig4/source_data/heatmaps/Fig4D_{group}_D{day}_CL_matrices.npz'

with np.load(path) as z:
    within[group, day] = {'Odd': z['map_a_display'], 'Even': z['map_b_display']}


### Read the source data

In [ ]:
day = 5

path = Path(data_root) / f'fig4/source_data/heatmaps/Fig4D_{group}_D{day}_CL_matrices.npz'

with np.load(path) as z:
    within[group, day] = {'Odd': z['map_a_display'], 'Even': z['map_b_display']}


### Read the source data

In [ ]:
a, b = (1, 2)

path = Path(data_root) / f'fig4/source_data/heatmaps/Fig4H_{group}_D{a}-D{b}_CL_matrices.npz'

with np.load(path) as z:
    adjacent[group, f'D{a}–D{b}'] = {'earlier': z['map_a_display'], 'later': z['map_b_display']}


### Read the source data

In [ ]:
a, b = (2, 3)

path = Path(data_root) / f'fig4/source_data/heatmaps/Fig4H_{group}_D{a}-D{b}_CL_matrices.npz'

with np.load(path) as z:
    adjacent[group, f'D{a}–D{b}'] = {'earlier': z['map_a_display'], 'later': z['map_b_display']}


### Read the source data

In [ ]:
a, b = (3, 4)

path = Path(data_root) / f'fig4/source_data/heatmaps/Fig4H_{group}_D{a}-D{b}_CL_matrices.npz'

with np.load(path) as z:
    adjacent[group, f'D{a}–D{b}'] = {'earlier': z['map_a_display'], 'later': z['map_b_display']}


### Read the source data

In [ ]:
a, b = (4, 5)

path = Path(data_root) / f'fig4/source_data/heatmaps/Fig4H_{group}_D{a}-D{b}_CL_matrices.npz'

with np.load(path) as z:
    adjacent[group, f'D{a}–D{b}'] = {'earlier': z['map_a_display'], 'later': z['map_b_display']}


### Fig5 b — draw and export

In [ ]:
fig = h.plot_fig4d_activity_heatmap(within, days=(1, 3, 5))[0]

exports += save_panel(fig, output, 'Fig5_b', 'odd_even_activity_maps', old_label='D', formats=formats, dpi=dpi)


### Fig5 g — draw and export

In [ ]:
fig = h.plot_fig4h_adjacent_activity_maps(adjacent)[0]

exports += save_panel(fig, output, 'Fig5_g', 'adjacent_day_activity_maps', old_label='H', formats=formats, dpi=dpi)


### Fig5 c — prepare panel inputs

In [ ]:
old, panel, table, time_col, time_order, label = ('E', 'Fig5_c', sources['within_pv_bin'], 'day', (1, 2, 3, 4, 5), 'PV reliability (r)')


### Fig5 c — draw and export

In [ ]:
fig = h.plot_spatial_bin_heatmap(table, panel=old, time_col=time_col, time_order=time_order, condition_col='condition', condition_value='CL', metric_label=label, vmin=h.PLOT_LIMITS[old][0], vmax=h.PLOT_LIMITS[old][1])[0]

exports += save_panel(fig, output, panel, 'PV_heatmap', old_label=old, formats=formats, dpi=dpi)


### Fig5 h — prepare panel inputs

In [ ]:
old, panel, table, time_col, time_order, label = ('I', 'Fig5_h', sources['inter_pv_bin'][sources['inter_pv_bin'].pair_mode.eq('adjacent')], 'day_pair_display', ('D1–D2', 'D2–D3', 'D3–D4', 'D4–D5'), 'PV stability (r)')


### Fig5 h — draw and export

In [ ]:
fig = h.plot_spatial_bin_heatmap(table, panel=old, time_col=time_col, time_order=time_order, condition_col='condition', condition_value='CL', metric_label=label, vmin=h.PLOT_LIMITS[old][0], vmax=h.PLOT_LIMITS[old][1])[0]

exports += save_panel(fig, output, panel, 'PV_heatmap', old_label=old, formats=formats, dpi=dpi)


### Fig5 d — prepare panel inputs

In [ ]:
old, panel, analysis, metric = ('F', 'Fig5_d', 'within', 'single_neuron')

key = ('within_single' if metric == 'single_neuron' else 'within_pv') if analysis == 'within' else 'inter_single' if metric == 'single_neuron' else 'inter_pv'

obs = sources[key]

if analysis != 'within':
    obs = obs[obs.pair_mode.eq(analysis)]

summary_name = f'{analysis}_{('single' if metric == 'single_neuron' else 'pv')}'


### Fig5 d — draw and export

In [ ]:
fig = h.plot_correlation_curve_panel(h._display_condition(obs), h._display_condition(descriptives[summary_name]), x_col='day' if analysis == 'within' else 'time_index', x_order=(1, 2, 3, 4, 5) if analysis == 'within' else (1, 2, 3, 4), panel_kind='neuron' if metric == 'single_neuron' else 'pv', xlabel='Day' if analysis == 'within' else 'Day pair', ylabel='Spatial activity correlation (r)' if metric == 'single_neuron' else 'PV correlation (r)', stat_annotations=h.build_annotations(statistics, analysis=analysis, metric=metric), show_interaction=True, ylim=h.PLOT_LIMITS[old])[0]

exports += save_panel(fig, output, panel, 'spatial_stability', formats=formats, dpi=dpi)


### Fig5 e — prepare panel inputs

In [ ]:
old, panel, analysis, metric = ('G', 'Fig5_e', 'within', 'pv')

key = ('within_single' if metric == 'single_neuron' else 'within_pv') if analysis == 'within' else 'inter_single' if metric == 'single_neuron' else 'inter_pv'

obs = sources[key]

if analysis != 'within':
    obs = obs[obs.pair_mode.eq(analysis)]

summary_name = f'{analysis}_{('single' if metric == 'single_neuron' else 'pv')}'


### Fig5 e — draw and export

In [ ]:
fig = h.plot_correlation_curve_panel(h._display_condition(obs), h._display_condition(descriptives[summary_name]), x_col='day' if analysis == 'within' else 'time_index', x_order=(1, 2, 3, 4, 5) if analysis == 'within' else (1, 2, 3, 4), panel_kind='neuron' if metric == 'single_neuron' else 'pv', xlabel='Day' if analysis == 'within' else 'Day pair', ylabel='Spatial activity correlation (r)' if metric == 'single_neuron' else 'PV correlation (r)', stat_annotations=h.build_annotations(statistics, analysis=analysis, metric=metric), show_interaction=True, ylim=h.PLOT_LIMITS[old])[0]

exports += save_panel(fig, output, panel, 'spatial_stability', formats=formats, dpi=dpi)


### Fig5 i — prepare panel inputs

In [ ]:
old, panel, analysis, metric = ('J', 'Fig5_i', 'adjacent', 'single_neuron')

key = ('within_single' if metric == 'single_neuron' else 'within_pv') if analysis == 'within' else 'inter_single' if metric == 'single_neuron' else 'inter_pv'

obs = sources[key]

if analysis != 'within':
    obs = obs[obs.pair_mode.eq(analysis)]

summary_name = f'{analysis}_{('single' if metric == 'single_neuron' else 'pv')}'


### Fig5 i — draw and export

In [ ]:
fig = h.plot_correlation_curve_panel(h._display_condition(obs), h._display_condition(descriptives[summary_name]), x_col='day' if analysis == 'within' else 'time_index', x_order=(1, 2, 3, 4, 5) if analysis == 'within' else (1, 2, 3, 4), panel_kind='neuron' if metric == 'single_neuron' else 'pv', xlabel='Day' if analysis == 'within' else 'Day pair', ylabel='Spatial activity correlation (r)' if metric == 'single_neuron' else 'PV correlation (r)', stat_annotations=h.build_annotations(statistics, analysis=analysis, metric=metric), show_interaction=True, ylim=h.PLOT_LIMITS[old])[0]

exports += save_panel(fig, output, panel, 'spatial_stability', formats=formats, dpi=dpi)


### Fig5 j — prepare panel inputs

In [ ]:
old, panel, analysis, metric = ('K', 'Fig5_j', 'adjacent', 'pv')

key = ('within_single' if metric == 'single_neuron' else 'within_pv') if analysis == 'within' else 'inter_single' if metric == 'single_neuron' else 'inter_pv'

obs = sources[key]

if analysis != 'within':
    obs = obs[obs.pair_mode.eq(analysis)]

summary_name = f'{analysis}_{('single' if metric == 'single_neuron' else 'pv')}'


### Fig5 j — draw and export

In [ ]:
fig = h.plot_correlation_curve_panel(h._display_condition(obs), h._display_condition(descriptives[summary_name]), x_col='day' if analysis == 'within' else 'time_index', x_order=(1, 2, 3, 4, 5) if analysis == 'within' else (1, 2, 3, 4), panel_kind='neuron' if metric == 'single_neuron' else 'pv', xlabel='Day' if analysis == 'within' else 'Day pair', ylabel='Spatial activity correlation (r)' if metric == 'single_neuron' else 'PV correlation (r)', stat_annotations=h.build_annotations(statistics, analysis=analysis, metric=metric), show_interaction=True, ylim=h.PLOT_LIMITS[old])[0]

exports += save_panel(fig, output, panel, 'spatial_stability', formats=formats, dpi=dpi)


### Fig5 k — prepare panel inputs

In [ ]:
old, panel, analysis, metric = ('L', 'Fig5_k', 'anchor', 'single_neuron')

key = ('within_single' if metric == 'single_neuron' else 'within_pv') if analysis == 'within' else 'inter_single' if metric == 'single_neuron' else 'inter_pv'

obs = sources[key]

if analysis != 'within':
    obs = obs[obs.pair_mode.eq(analysis)]

summary_name = f'{analysis}_{('single' if metric == 'single_neuron' else 'pv')}'


### Fig5 k — draw and export

In [ ]:
fig = h.plot_correlation_curve_panel(h._display_condition(obs), h._display_condition(descriptives[summary_name]), x_col='day' if analysis == 'within' else 'time_index', x_order=(1, 2, 3, 4, 5) if analysis == 'within' else (1, 2, 3, 4), panel_kind='neuron' if metric == 'single_neuron' else 'pv', xlabel='Day' if analysis == 'within' else 'Day pair', ylabel='Spatial activity correlation (r)' if metric == 'single_neuron' else 'PV correlation (r)', stat_annotations=h.build_annotations(statistics, analysis=analysis, metric=metric), show_interaction=True, ylim=h.PLOT_LIMITS[old])[0]

exports += save_panel(fig, output, panel, 'spatial_stability', formats=formats, dpi=dpi)


### Fig5 l — prepare panel inputs

In [ ]:
old, panel, analysis, metric = ('M', 'Fig5_l', 'anchor', 'pv')

key = ('within_single' if metric == 'single_neuron' else 'within_pv') if analysis == 'within' else 'inter_single' if metric == 'single_neuron' else 'inter_pv'

obs = sources[key]

if analysis != 'within':
    obs = obs[obs.pair_mode.eq(analysis)]

summary_name = f'{analysis}_{('single' if metric == 'single_neuron' else 'pv')}'


### Fig5 l — draw and export

In [ ]:
fig = h.plot_correlation_curve_panel(h._display_condition(obs), h._display_condition(descriptives[summary_name]), x_col='day' if analysis == 'within' else 'time_index', x_order=(1, 2, 3, 4, 5) if analysis == 'within' else (1, 2, 3, 4), panel_kind='neuron' if metric == 'single_neuron' else 'pv', xlabel='Day' if analysis == 'within' else 'Day pair', ylabel='Spatial activity correlation (r)' if metric == 'single_neuron' else 'PV correlation (r)', stat_annotations=h.build_annotations(statistics, analysis=analysis, metric=metric), show_interaction=True, ylim=h.PLOT_LIMITS[old])[0]

exports += save_panel(fig, output, panel, 'spatial_stability', formats=formats, dpi=dpi)


## 5. Export record

Panel files and any refitted model tables are saved under the figure output directory. The record identifies the selected manuscript workflow and whether statistics were refitted.

In [ ]:
write_run_record(output / "run.json", figure="5", refit=refit, exports=exports)
display(pd.DataFrame({"exported_file": [str(Path(p).relative_to(output)) for p in exports]}))
